# Legal language model with sparse Mixture-of-Experts: CS781 Phase 1, architecture 5

A self-contained notebook: it streams the corpus, trains the tokenizer, trains the model **from random weights**, writes the Codabench submission. **No datasets need to be attached.**

| Stage | Object | What it does |
|---|---|---|
| 1 | `Settings` | every hyper-parameter in one dataclass (`smoke=True` → tiny rehearsal) |
| 2 | `src/` | `a1_tokenizer.py` (your A1 BPE), `rope.py` (your A3 RoPE), `space_bpe.py`, `moe_lm.py` (the model) |
| 3 | `LegalCorpus` | streams the 4 sources from Hugging Face, document-level split, dev set shaped like the test set |
| 4 | `TokenStore` | trains the space-attached BPE, encodes `train.bin` / `val.bin` in parallel |
| 5 | `Trainer` | AdamW + cosine schedule, fp16, both GPUs (DataParallel), EMA weights, checkpoints, resume, time budget |
| 6 | `Generator` | KV-cache sampling (temperature / top-k / top-p), batched on both GPUs, exactly 128 words |
| 7 | submission helpers | `prediction.json` → `submission.zip` (verified), dev metrics, one download bundle |

**The model:** a decoder-only Transformer whose feed-forward layers are **sparse mixtures of 8 SwiGLU experts with top-2 routing**, plus **grouped-query attention** (12 query heads / 4 KV heads), RoPE and RMSNorm. ≈252M parameters in total, ≈82M used per token.

**Run it:** GPU **T4 x2**, Internet **On**, Secret `HF_TOKEN`.
1. `smoke=True` (about 15 min). Check: the initial loss is ≈ ln(V), it falls, every answer is 128 words, and the zip is verified.
2. `smoke=False`, then **Save & Run All (Commit)**. Data preparation takes ~20 min, training ~8.5–9.5 h (read `tok/s` / `ETA` in the log). If your GPU quota is smaller, lower `session_hours`: training then stops by itself and the submission is still produced. The bundle keeps `last.pt`, so `resume=` can finish the run in a second session.

In [ ]:
!pip -q install -U "huggingface_hub>=0.23" ijson rouge-score sacrebleu bert-score

In [ ]:
# ── 1 · Settings ───────────────────────────────────────────────────────────────
from dataclasses import dataclass, asdict, replace
import os, sys, json, math, time, random, zipfile, shutil, pickle, copy, threading, queue, importlib
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F


@dataclass
class Settings:
    run: str = "moe_lm"
    smoke: bool = True                     # True = tiny rehearsal of the whole notebook; False = the real run
    # corpus
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    sources: tuple = ("acts", "high_courts", "supreme_court", "tribunals")
    train_tokens: int = 655_361_025        # = steps x 32,768 + 1,025 -> exactly one pass, nothing repeated
    val_tokens_per_source: int = 500_000
    val_doc_frac: float = 0.02             # documents held out per source (never trained on)
    dev_prompts: int = 100                 # 512-word prompt -> next 128 words, from held-out documents
    chars_per_token_guess: float = 4.8     # only sizes the download; the real value is printed later
    # tokenizer
    vocab: int = 8000
    tokenizer_chars: int = 30_000_000
    # model
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    n_experts: int = 8
    top_k: int = 2
    expert_hidden: int = 1024
    ctx: int = 1024
    aux_coef: float = 0.01
    # optimisation
    micro_batch: int = 8
    accum: int = 4                         # 8 x 4 x 1024 = 32,768 tokens per optimizer step
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup: int = 800
    steps: int = 20_000
    weight_decay: float = 0.1
    clip: float = 1.0
    ema: float = 0.999                     # 0 = off
    eval_every: int = 2000
    eval_windows: int = 320                # fixed validation windows, split equally over the sources
    save_every: int = 500
    session_hours: float = 11.5            # Kaggle stops a session at 12 h
    reserve_hours: float = 0.8             # kept for generation + zip after training
    resume: str = None                     # path to a last.pt to continue an interrupted run
    # generation
    words: int = 128
    temperature: float = 0.8
    top_k_sample: int = 50
    top_p: float = 0.95
    gen_batch: int = 16
    prompt_room: int = 256                 # prompt keeps at most ctx - prompt_room tokens (room to generate)
    max_new: int = 1024
    seed: int = 1337
    out: str = "/kaggle/working"

    def rehearsal(self):
        return replace(self, run=self.run + "_smoke", train_tokens=120 * 8 * 256 + 257, val_tokens_per_source=12_500,
                       val_doc_frac=0.05, dev_prompts=16, vocab=2000, tokenizer_chars=1_000_000, d_model=256,
                       n_layers=4, n_heads=4, n_kv_heads=2, expert_hidden=256, ctx=256, accum=1, warmup=20,
                       steps=120, eval_every=40, eval_windows=40, save_every=40, prompt_room=96)


S = Settings()
S = S.rehearsal() if S.smoke else S
CLOCK = time.time()
hours = lambda: (time.time() - CLOCK) / 3600
CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda:0" if CUDA else "cpu")
N_GPU = torch.cuda.device_count() if CUDA else 0
RUN_DIR = os.path.join(S.out, S.run)
DATA_DIR = os.path.join(RUN_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
random.seed(S.seed); np.random.seed(S.seed); torch.manual_seed(S.seed)


def autocast():
    return torch.autocast(device_type="cuda" if CUDA else "cpu", dtype=torch.float16, enabled=CUDA)


print(f"run={S.run} | smoke={S.smoke} | device={DEVICE} x{max(N_GPU, 1)} | tokens/step="
      f"{S.micro_batch * S.accum * S.ctx:,} | steps={S.steps:,}")

In [ ]:
# Hugging Face login (the dataset is gated; accept its terms once on the website)
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN")); print("logged in (Kaggle secret)")
except Exception:
    import getpass
    login(token=os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token: ")); print("logged in")

## 2 · Source files
`a1_tokenizer.py` and `rope.py` are your assignment code, unchanged. `space_bpe.py` extends the A1 tokenizer: a leading space is attached to the next word, and the trainer is faster but gives the same merges. `moe_lm.py` is the model.

In [ ]:
os.makedirs('src', exist_ok=True)

In [ ]:
%%writefile src/a1_tokenizer.py
"""Assignment-1 byte-level BPE tokenizer, retrained on the Assignment-3 corpus.

Byte-level BPE (A-1 Q5 spec): byte symbols so no <unk> is needed, reserved tokens
<pad> <unk> <s> </s> at ids 0..3, each merge takes the most frequent adjacent pair
(ties broken by the lexicographically smallest merged bytes), and merges never cross
a pre-token (word / whitespace) boundary so decode() reconstructs the text exactly.
"""
import re
from collections import Counter

RESERVED = ("<pad>", "<unk>", "<s>", "</s>")
BASE = len(RESERVED)
SPLIT = re.compile(r"\S+|\s+")


def merge_sequence(ids, pair, new_id):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)


class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.ranks = {}
        self.itos = {}
        self.vocab_size = 0
        self.cache = {}

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))

        counts = Counter(SPLIT.findall(text))
        corpus = Counter()
        for word, freq in counts.items():
            ids = tuple(BASE + byte for byte in word.encode("utf-8"))
            corpus[ids] += freq

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges:
            pair_freq = Counter()
            for ids, freq in corpus.items():
                for j in range(len(ids) - 1):
                    pair_freq[(ids[j], ids[j + 1])] += freq
            if not pair_freq:
                break
            top = max(pair_freq.values())
            if top < 2:
                break
            best_pairs = [p for p, f in pair_freq.items() if f == top]
            pair = min(best_pairs, key=lambda p: self.itos[p[0]] + self.itos[p[1]])

            self.itos[next_id] = self.itos[pair[0]] + self.itos[pair[1]]
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)

            new_corpus = Counter()
            for ids, freq in corpus.items():
                new_corpus[merge_sequence(ids, pair, next_id)] += freq
            corpus = new_corpus
            next_id += 1

        self.vocab_size = BASE + 256 + len(self.merges)

    def encode_chunk(self, raw_bytes):
        if raw_bytes in self.cache:
            return self.cache[raw_bytes]
        ids = [BASE + byte for byte in raw_bytes]
        while len(ids) >= 2:
            candidates = {}
            for j in range(len(ids) - 1):
                pair = (ids[j], ids[j + 1])
                if pair in self.ranks:
                    candidates[pair] = self.ranks[pair]
            if not candidates:
                break
            pair = min(candidates, key=candidates.get)
            ids = list(merge_sequence(ids, pair, self.merges[pair]))
        self.cache[raw_bytes] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids

    def decode(self, ids):
        data = b""
        for i in ids:
            data += self.itos.get(i, b"")
        return data.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self.vocab_size


def train_tokenizer(train_text, vocab_size):
    tokenizer = BPETokenizer()
    tokenizer.train(train_text, vocab_size)
    return tokenizer

In [ ]:
%%writefile src/rope.py
# TODO 3.1: rotary positional embeddings from scratch (split-half convention).
import torch


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([angles, angles], dim=-1)                               # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
%%writefile src/space_bpe.py
# Space-attached byte-level BPE (GPT-2 style pre-tokenisation) built on your A1 tokenizer.
#
# Difference from A1: A1 splits text into  \S+ | \s+  so every space is its own token
# (~2.3 tokens per word). Here a single leading space is attached to the following word
# (" the", " Court"), so a word usually costs ~1.3 tokens and a 512-word prompt fits in the
# 1024-token context. Merge rule is unchanged (most frequent pair, ties -> lexicographically
# smallest merged bytes, never across a pre-token boundary), so decode() is still exact.
# train() is an incremental version of the A1 algorithm: it only re-counts pairs in words that
# contain the merged pair (plus a heap), which gives the same merges but runs minutes, not hours.
import heapq
import re
from collections import Counter, defaultdict
from a1_tokenizer import BPETokenizer, BASE, merge_sequence

# " ?\S+"      a word with at most one leading space
# "\s+(?!\S)"  a whitespace run, leaving its last space for the next word
# "\s+"        trailing whitespace at the very end
SPACE_SPLIT = re.compile(r" ?\S+|\s+(?!\S)|\s+")


class SpaceBPETokenizer(BPETokenizer):
    space_prefix = True          # tells the pipeline not to append a trailing space to prompts

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))
        counts = Counter(SPACE_SPLIT.findall(text))
        words = [tuple(BASE + b for b in w.encode("utf-8")) for w in counts]
        freqs = list(counts.values())

        pair_counts = defaultdict(int)          # pair -> total frequency
        where = defaultdict(set)                # pair -> indices of words containing it
        for wi, ids in enumerate(words):
            for pair in zip(ids, ids[1:]):
                pair_counts[pair] += freqs[wi]
                where[pair].add(wi)
        merged = lambda p: self.itos[p[0]] + self.itos[p[1]]
        heap = [(-c, merged(p), p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges and heap:
            neg_c, _, pair = heapq.heappop(heap)
            c = pair_counts.get(pair, 0)
            if c != -neg_c or c <= 0:
                continue                         # stale heap entry
            if c < 2:
                break
            self.itos[next_id] = merged(pair)
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)
            delta = defaultdict(int)
            for wi in where.pop(pair, ()):
                old = words[wi]
                new = merge_sequence(old, pair, next_id)
                if new == old:
                    continue
                f = freqs[wi]
                for p in zip(old, old[1:]):
                    delta[p] -= f
                for p in zip(new, new[1:]):
                    delta[p] += f
                    where[p].add(wi)
                words[wi] = new
            pair_counts.pop(pair, None)
            for p, d in delta.items():
                if d == 0 or p == pair:
                    continue
                pair_counts[p] += d
                if pair_counts[p] > 0:
                    heapq.heappush(heap, (-pair_counts[p], merged(p), p))
            next_id += 1
        self.vocab_size = BASE + 256 + len(self.merges)

    def encode(self, text):
        ids = []
        for chunk in SPACE_SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids


def train_space_tokenizer(train_text, vocab_size):
    tok = SpaceBPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [ ]:
%%writefile src/moe_lm.py
"""Sparse Mixture-of-Experts language model (decoder-only) with grouped-query attention.

  token ids -> embedding -> N x [ RMSNorm -> GQA self-attention (RoPE, causal) -> residual
                                  RMSNorm -> MoE feed-forward (router + top-k of E SwiGLU experts) -> residual ]
           -> RMSNorm -> tied output projection -> next-token logits

MoE feed-forward: a linear router scores the E experts for every token; the token is sent to its k best
experts and their outputs are added, weighted by the router probabilities renormalised over those k.
Only k/E of the expert parameters are used per token (sparse, conditional computation).
Load balancing (Switch Transformer): aux = E * sum_e f_e * P_e, where f_e is the share of routed slots
that went to expert e and P_e its mean router probability; aux = 1 when perfectly balanced.
GQA: n_heads query heads share n_kv_heads key/value heads -> the KV cache is n_heads/n_kv_heads smaller.
"""
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope          # your A3 RoPE (split-half convention)


@dataclass
class MoEConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    n_experts: int = 8
    top_k: int = 2
    expert_hidden: int = 1024
    max_seq_len: int = 1024
    aux_coef: float = 0.01


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight


class Expert(nn.Module):
    """One SwiGLU feed-forward network: W2( SiLU(W1 x) * W3 x )."""

    def __init__(self, d, hidden):
        super().__init__()
        self.w1 = nn.Linear(d, hidden, bias=False)
        self.w3 = nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class SparseMoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_experts, self.top_k = cfg.n_experts, cfg.top_k
        self.router = nn.Linear(cfg.d_model, cfg.n_experts, bias=False)
        self.experts = nn.ModuleList(Expert(cfg.d_model, cfg.expert_hidden) for _ in range(cfg.n_experts))
        self.aux = None          # load-balancing loss of the last forward pass
        self.load = None         # share of routed slots per expert (diagnostics)

    def forward(self, x):
        shape = x.shape
        tokens = x.reshape(-1, shape[-1])
        probs = F.softmax(self.router(tokens).float(), dim=-1)               # (N, E)
        weight, chosen = probs.topk(self.top_k, dim=-1)                      # (N, k)
        weight = weight / weight.sum(dim=-1, keepdim=True)
        mixed = torch.zeros_like(tokens)
        for e, expert in enumerate(self.experts):
            row, slot = torch.where(chosen == e)
            if row.numel():
                out = expert(tokens[row]) * weight[row, slot, None].to(tokens.dtype)
                mixed.index_add_(0, row, out.to(mixed.dtype))
        share = F.one_hot(chosen, self.n_experts).float().mean(dim=(0, 1))   # f_e
        self.aux = self.n_experts * (share * probs.mean(dim=0)).sum()
        self.load = share.detach()
        return mixed.view(shape)


class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.g, self.dh = cfg.n_heads, cfg.n_kv_heads, cfg.d_model // cfg.n_heads
        self.wq = nn.Linear(cfg.d_model, self.h * self.dh, bias=False)
        self.wkv = nn.Linear(cfg.d_model, 2 * self.g * self.dh, bias=False)
        self.wo = nn.Linear(self.h * self.dh, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, past=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.dh).transpose(1, 2)
        k, v = self.wkv(x).view(B, T, 2, self.g, self.dh).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos.to(q.dtype), sin.to(q.dtype)), apply_rope(k, cos.to(k.dtype), sin.to(k.dtype))
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        present = (k, v)
        rep = self.h // self.g
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        if past is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:                                       # the T new queries are the last T positions
            S = k.size(2)
            allowed = torch.arange(S, device=x.device)[None, :] <= torch.arange(S - T, S, device=x.device)[:, None]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
        return self.wo(y.transpose(1, 2).reshape(B, T, -1)), present


class Layer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm, self.attn = RMSNorm(cfg.d_model), GroupedQueryAttention(cfg)
        self.ffn_norm, self.ffn = RMSNorm(cfg.d_model), SparseMoE(cfg)

    def forward(self, x, cos, sin, past=None):
        a, present = self.attn(self.attn_norm(x), cos, sin, past)
        x = x + a
        return x + self.ffn(self.ffn_norm(x)), present


class MoELanguageModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList(Layer(cfg) for _ in range(cfg.n_layers))
        self.final_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        cos, sin = build_rope_cache(cfg.max_seq_len, cfg.d_model // cfg.n_heads)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def param_counts(self):
        total = sum(p.numel() for p in self.parameters())
        one_expert = sum(p.numel() for p in self.layers[0].ffn.experts[0].parameters())
        idle = self.cfg.n_layers * (self.cfg.n_experts - self.cfg.top_k) * one_expert
        return total, total - idle

    def _stack(self, ids, start=0, past=None):
        T = ids.size(1)
        cos, sin = self.cos[start:start + T], self.sin[start:start + T]
        x, presents = self.embed(ids), []
        for i, layer in enumerate(self.layers):
            x, p = layer(x, cos, sin, None if past is None else past[i])
            presents.append(p)
        return self.final_norm(x), presents

    def hidden_states(self, ids):
        return self._stack(ids)[0]

    def forward(self, ids):
        return self.lm_head(self._stack(ids)[0])

    def loss(self, x, y):
        logits = self.lm_head(self._stack(x)[0])
        ce = F.cross_entropy(logits.float().flatten(0, 1), y.flatten())
        if self.training and self.cfg.aux_coef:
            ce = ce + self.cfg.aux_coef * torch.stack([l.ffn.aux for l in self.layers]).mean()
        return ce

    def expert_load(self):
        return torch.stack([l.ffn.load for l in self.layers])

    @torch.no_grad()
    def prefill(self, ids):
        h, past = self._stack(ids)
        return self.lm_head(h[:, -1]), past

    @torch.no_grad()
    def step(self, ids, past, pos):
        h, past = self._stack(ids, pos, past)
        return self.lm_head(h[:, -1]), past

In [ ]:
sys.path.insert(0, os.path.abspath("src"))
import a1_tokenizer, rope, space_bpe, moe_lm
for _m in (a1_tokenizer, rope, space_bpe, moe_lm): importlib.reload(_m)
from space_bpe import train_space_tokenizer
from moe_lm import MoEConfig, MoELanguageModel
SPECIAL = [0, 1, 2, 3]          # <pad> <unk> <s> </s> in the A1 tokenizer
EOS = 3
print("source files imported")

## 3 · Corpus

In [ ]:
import ijson
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download


class LegalCorpus:
    """Streams the Phase-1 sources (no full download), splits by document, builds a test-shaped dev set."""

    def __init__(self, s):
        self.s = s
        files = [f for f in HfApi().list_repo_files(s.hf_repo, repo_type="dataset") if f.startswith("Phase_1")]
        self.train_files = {src: next(f for f in files if "/Train/" in f and src in f.lower()) for src in s.sources}
        self.test_file = next(f for f in files if "/Test/" in f and f.endswith(".json"))
        self.fs = HfFileSystem()

    def _open(self, path):
        return self.fs.open(f"datasets/{self.s.hf_repo}/{path}", "rb", block_size=16 << 20)

    @classmethod
    def _texts(cls, obj):
        if isinstance(obj, str):
            if obj.strip(): yield obj
        elif isinstance(obj, dict):
            for key in ("text", "content", "body", "document", "judgment", "raw_text"):
                if isinstance(obj.get(key), str) and obj[key].strip():
                    yield obj[key]; return
            for v in obj.values(): yield from cls._texts(v)
        elif isinstance(obj, list):
            for v in obj: yield from cls._texts(v)

    def stream(self, src, budget, skip=0):
        """Documents of one source in file order: skip the first `skip` characters, then collect `budget`."""
        path = self.train_files[src]
        for attempt in range(1, 4):
            try:
                with self._open(path) as f:
                    head = f.read(64).lstrip()[:1]
                docs, got, skipped = [], 0, 0
                with self._open(path) as f:
                    items = ijson.kvitems(f, "") if head == b"{" else ijson.items(f, "item")
                    for item in items:
                        for t in self._texts(item[1] if head == b"{" else item):
                            if skipped < skip:
                                skipped += len(t); continue
                            docs.append(t); got += len(t)
                            if got >= budget: return docs
                return docs
            except Exception as e:
                print(f"  {src}: stream error {type(e).__name__} ({e}); retry {attempt}/3"); time.sleep(15)
        raise RuntimeError(f"cannot stream {path}")

    def build(self):
        s = self.s
        total = s.train_tokens * s.chars_per_token_guess / (1 - s.val_doc_frac)
        quota = int(total / len(s.sources))
        docs, used = {}, {}
        for src in s.sources:
            t0 = time.time(); docs[src] = self.stream(src, quota); used[src] = sum(map(len, docs[src]))
            print(f"{src:>14}: {len(docs[src]):>8,} docs {used[src]:>14,} chars"
                  + ("  (source exhausted)" if used[src] < quota else "") + f"  {time.time()-t0:.0f}s")
        short, open_src = total - sum(used.values()), [x for x in s.sources if used[x] >= quota]
        if short > 0.02 * total and open_src:                    # a small source ran out: top up from the others
            for src in open_src:
                more = self.stream(src, short / len(open_src), skip=used[src])
                docs[src] += more; used[src] += sum(map(len, more))
                print(f"   top-up {src}: +{len(more):,} docs")
        rng = random.Random(s.seed)
        train, held = [], {}
        for src in s.sources:
            rng.shuffle(docs[src])
            n = max(1, round(len(docs[src]) * s.val_doc_frac))
            held[src] = docs[src][:n]
            train += [(src, d) for d in docs[src][n:]]
        rng.shuffle(train)
        print(f"train docs {len(train):,} | held-out docs {sum(map(len, held.values())):,}")
        return train, held

    @staticmethod
    def dev_set(held, n, seed, prompt_words=512, cont_words=128):
        """Same shape as the test set: a 512-word prompt and the next 128 words, n/len(held) per source."""
        prompts, refs, rng = {}, {}, random.Random(seed)
        for src, docs in held.items():
            pool = [d.split() for d in docs]
            pool = [w for w in pool if len(w) >= prompt_words + cont_words] or [sum(pool, [])]
            for _ in range(max(1, n // len(held))):
                w = rng.choice(pool); i = rng.randint(0, len(w) - prompt_words - cont_words); key = str(len(prompts))
                prompts[key] = {"text": " ".join(w[i:i + prompt_words]), "source": src}
                refs[key] = {"text": " ".join(w[i + prompt_words:i + prompt_words + cont_words]), "source": src}
        return prompts, refs

    def test_prompts(self):
        with open(hf_hub_download(self.s.hf_repo, self.test_file, repo_type="dataset"), encoding="utf-8") as f:
            raw = json.load(f)
        pick = lambda v: v if isinstance(v, str) else next((v[k] for k in ("text", "content", "prompt", "body")
                                                             if isinstance(v.get(k), str)), "")
        return {k: {"text": pick(v)} for k, v in raw.items()}

## 4 · Tokenizer and token files

In [ ]:
from multiprocessing import Pool
import hashlib

_TOKENIZER = None


def _encode_doc(doc):
    ids = _TOKENIZER.encode(doc); ids.append(EOS)
    return np.asarray(ids, dtype=np.uint16)


class TokenStore:
    """Owns the tokenizer and the uint16 token files (train.bin, val.bin) plus the dev set."""

    def __init__(self, s, folder):
        self.s, self.folder = s, folder
        self.path = lambda name: os.path.join(folder, name)

    def ready(self):
        return os.path.exists(self.path("meta.json"))

    @staticmethod
    def fingerprint(tok):
        return hashlib.md5(json.dumps(sorted([a, b, c] for (a, b), c in tok.merges.items())).encode()).hexdigest()[:12]

    def _encode(self, docs, cap):
        out, n, lens = np.empty(cap, dtype=np.uint16), 0, []
        with Pool(os.cpu_count() or 2) as pool:
            for arr in pool.imap(_encode_doc, docs, chunksize=8):
                k = min(len(arr), cap - n); out[n:n + k] = arr[:k]; n += k; lens.append(k)
                if n >= cap: break
        return out[:n], lens

    def build(self, train, held, dev):
        global _TOKENIZER
        s, t0 = self.s, time.time()
        text, n = [], 0
        for _, d in train:
            text.append(d); n += len(d)
            if n >= s.tokenizer_chars: break
        _TOKENIZER = train_space_tokenizer("\n\n".join(text)[:s.tokenizer_chars], s.vocab)
        with open(self.path("tokenizer.pkl"), "wb") as f:
            cache, _TOKENIZER.cache = _TOKENIZER.cache, {}
            pickle.dump(_TOKENIZER, f); _TOKENIZER.cache = cache
        print(f"tokenizer: vocab {_TOKENIZER.get_vocab_size()} trained in {time.time()-t0:.0f}s")
        parts, ranges, pos = [], {}, 0
        for src, docs in held.items():
            arr, _ = self._encode(docs, s.val_tokens_per_source)
            parts.append(arr); ranges[src] = [pos, pos + len(arr)]; pos += len(arr)
        val = np.concatenate(parts)
        train_ids, lens = self._encode([d for _, d in train], s.train_tokens)
        by_src = {src: 0 for src in s.sources}
        for (src, _), k in zip(train, lens): by_src[src] += k
        chars = sum(len(d) for (_, d), _k in zip(train, lens))
        train_ids.tofile(self.path("train.bin")); val.tofile(self.path("val.bin"))
        json.dump(dev[0], open(self.path("dev_prompts.json"), "w"), ensure_ascii=False)
        json.dump(dev[1], open(self.path("dev_refs.json"), "w"), ensure_ascii=False)
        meta = dict(train_tokens=int(len(train_ids)), val_tokens=int(len(val)), val_ranges=ranges,
                    train_tokens_by_source=by_src, chars_per_token=round(chars / max(1, len(train_ids)), 3),
                    tokenizer=self.fingerprint(_TOKENIZER), vocab=_TOKENIZER.get_vocab_size())
        json.dump(meta, open(self.path("meta.json"), "w"), indent=1)
        print(f"encoded in {time.time()-t0:.0f}s:", {k: v for k, v in meta.items() if k != "val_ranges"})

    def load(self):
        tok = pickle.load(open(self.path("tokenizer.pkl"), "rb"))
        meta = json.load(open(self.path("meta.json")))
        assert self.fingerprint(tok) == meta["tokenizer"], "tokenizer and token files do not match"
        train = np.memmap(self.path("train.bin"), dtype=np.uint16, mode="r")
        val = np.memmap(self.path("val.bin"), dtype=np.uint16, mode="r")
        dev = (json.load(open(self.path("dev_prompts.json"), encoding="utf-8")),
               json.load(open(self.path("dev_refs.json"), encoding="utf-8")))
        return tok, train, val, meta, dev

## 5 · Trainer

In [ ]:
class _LossOnReplica(nn.Module):
    """DataParallel runs this once per GPU; each replica returns its own loss."""

    def __init__(self, model):
        super().__init__(); self.model = model

    def forward(self, x, y):
        with autocast():
            return self.model.loss(x, y).float().reshape(1)


class Trainer:
    def __init__(self, s, model, train, val, meta, run_dir, fingerprint):
        self.s, self.model, self.train, self.val, self.dir, self.fp = s, model.to(DEVICE), train, val, run_dir, fingerprint
        groups = [{"params": [p for p in model.parameters() if p.dim() >= 2], "weight_decay": s.weight_decay},
                  {"params": [p for p in model.parameters() if p.dim() < 2], "weight_decay": 0.0}]
        self.opt = torch.optim.AdamW(groups, lr=s.lr, betas=(0.9, 0.95))
        self.scaler = torch.amp.GradScaler("cuda", enabled=CUDA)
        self.rng = torch.Generator().manual_seed(s.seed)
        self.ema = ({k: v.detach().float().clone() for k, v in model.state_dict().items() if v.dtype.is_floating_point}
                    if s.ema else None)
        self.step, self.best, self.history = 0, float("inf"), []
        gpus = N_GPU if N_GPU > 1 and s.accum % N_GPU == 0 else 1
        self.parallel = nn.DataParallel(_LossOnReplica(model), device_ids=list(range(gpus))) if gpus > 1 else None
        self.micro, self.accum = s.micro_batch * gpus, s.accum // gpus
        per = max(1, s.eval_windows // len(meta["val_ranges"]))
        self.windows = {src: np.linspace(a, b - s.ctx - 2, per).astype(int).tolist()
                        for src, (a, b) in meta["val_ranges"].items() if b - a > s.ctx + 2}
        print(f"trainer: {gpus} GPU(s), micro-batch {self.micro} x {self.accum} accumulation | EMA {s.ema or 'off'}")

    # -- data ------------------------------------------------------------------------------------------
    def _batch(self, data, starts=None):
        T = self.s.ctx
        if starts is None:
            starts = torch.randint(0, len(data) - T - 1, (self.micro,), generator=self.rng).tolist()
        x = np.stack([data[i:i + T] for i in starts]).astype(np.int64)
        y = np.stack([data[i + 1:i + T + 1] for i in starts]).astype(np.int64)
        return torch.from_numpy(x).to(DEVICE), torch.from_numpy(y).to(DEVICE)

    def lr_at(self, step):
        s = self.s
        if step < s.warmup: return s.lr * step / s.warmup
        p = min(1.0, (step - s.warmup) / max(1, s.steps - s.warmup))
        return s.min_lr + 0.5 * (s.lr - s.min_lr) * (1 + math.cos(math.pi * p))

    # -- evaluation ------------------------------------------------------------------------------------
    @torch.no_grad()
    def val_loss(self, starts):
        was = self.model.training; self.model.eval(); total = 0.0
        for i in range(0, len(starts), self.s.micro_batch):
            x, y = self._batch(self.val, starts[i:i + self.s.micro_batch])
            with autocast():
                total += self.model.loss(x, y).item() * x.size(0)
        self.model.train(was)
        return total / len(starts)

    def val_by_source(self):
        return {src: round(self.val_loss(w), 4) for src, w in self.windows.items()}

    def _swap(self, new):
        cur = self.model.state_dict(); old = {k: cur[k].detach().clone() for k in new}
        with torch.no_grad():
            for k, v in new.items(): cur[k].copy_(v.to(cur[k].dtype))
        return old

    def evaluate(self):
        """Loss of the raw weights and of the EMA weights; the better one is left loaded (caller restores raw)."""
        all_w = sum(self.windows.values(), [])
        raw = self.val_loss(all_w); ema, backup, which = None, None, "raw"
        if self.ema is not None:
            backup = self._swap(self.ema); ema = self.val_loss(all_w)
            if ema < raw: which = "ema"
            else: self._swap(backup); backup = None
        return min(raw, ema if ema is not None else raw), raw, ema, which, backup

    # -- checkpoints -----------------------------------------------------------------------------------
    def _save(self, obj, name):
        path = os.path.join(self.dir, name); torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

    def save_last(self):
        self._save({"model": self.model.state_dict(), "opt": self.opt.state_dict(), "scaler": self.scaler.state_dict(),
                    "ema": self.ema, "step": self.step, "best": self.best, "history": self.history,
                    "rng": self.rng.get_state(), "torch_rng": torch.get_rng_state(), "tokenizer": self.fp,
                    "settings": asdict(self.s), "config": asdict(self.model.cfg)}, "last.pt")

    def resume(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=False)
        assert ck["tokenizer"] == self.fp, "checkpoint was trained with a different tokenizer"
        self.model.load_state_dict(ck["model"]); self.opt.load_state_dict(ck["opt"]); self.scaler.load_state_dict(ck["scaler"])
        if self.ema is not None and ck.get("ema"):
            for k in self.ema: self.ema[k].copy_(ck["ema"][k])
        self.rng.set_state(ck["rng"].cpu()); torch.set_rng_state(ck["torch_rng"].cpu())
        self.step, self.best, self.history = ck["step"], ck["best"], ck["history"]
        src_best = os.path.join(os.path.dirname(path), "best.pt")
        if os.path.exists(src_best) and not os.path.exists(os.path.join(self.dir, "best.pt")):
            shutil.copy(src_best, os.path.join(self.dir, "best.pt"))
        print(f"resumed at step {self.step} (best val {self.best:.3f})")

    # -- training loop ---------------------------------------------------------------------------------
    def fit(self):
        s, model = self.s, self.model
        model.train(); deadline = s.session_hours - s.reserve_hours
        tokens_per_step = self.micro * self.accum * s.ctx
        log_every = max(1, s.eval_every // 5); t_log = time.time(); running = torch.zeros((), device=DEVICE); seen = 0
        while self.step < s.steps:
            self.step += 1
            lr = self.lr_at(self.step)
            for g in self.opt.param_groups: g["lr"] = lr
            self.opt.zero_grad(set_to_none=True)
            for _ in range(self.accum):
                x, y = self._batch(self.train)
                if self.parallel is not None:
                    loss = self.parallel(x, y).mean() / self.accum
                else:
                    with autocast():
                        loss = model.loss(x, y) / self.accum
                self.scaler.scale(loss).backward(); running += loss.detach()
            self.scaler.unscale_(self.opt)
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), s.clip)
            self.scaler.step(self.opt); self.scaler.update(); seen += 1
            if self.ema is not None:
                with torch.no_grad():
                    sd = model.state_dict()
                    for k, e in self.ema.items(): e.mul_(s.ema).add_(sd[k].float(), alpha=1 - s.ema)
            if self.step % log_every == 0:
                dt = time.time() - t_log; t_log = time.time()
                print(f"step {self.step:>6} | lr {lr:.2e} | train {(running / seen).item():.3f} | gnorm {float(gnorm):.2f} | "
                      f"{log_every * tokens_per_step / dt:,.0f} tok/s | ETA {(s.steps - self.step) * dt / log_every / 60:.0f} min")
                running.zero_(); seen = 0
            if self.step % s.eval_every == 0 or self.step == s.steps:
                loss, raw, ema, which, backup = self.evaluate()
                self.history.append(dict(step=self.step, lr=lr, val=loss, raw=raw, ema=ema, hours=round(hours(), 2)))
                note = ""
                if loss < self.best:
                    self.best = loss; note = "  <- best, saved"
                    self._save({"model": model.state_dict(), "step": self.step, "val": loss, "weights": which,
                                "tokenizer": self.fp, "config": asdict(model.cfg)}, "best.pt")
                if backup is not None: self._swap(backup)
                print(f"  eval {self.step}: val {loss:.3f} (raw {raw:.3f}" + (f" / ema {ema:.3f}" if ema else "")
                      + f" -> {which}){note}")
                json.dump(self.history, open(os.path.join(self.dir, "history.json"), "w"), indent=1)
            if self.step % s.save_every == 0: self.save_last()
            if hours() > deadline:
                print(f"time budget reached at step {self.step} ({hours():.1f} h) - stopping; resume later with resume=")
                break
        self.save_last()
        if not os.path.exists(os.path.join(self.dir, "best.pt")):
            self._save({"model": model.state_dict(), "step": self.step, "val": self.val_loss(sum(self.windows.values(), [])),
                        "weights": "raw", "tokenizer": self.fp, "config": asdict(model.cfg)}, "best.pt")

## 6 · Generator

In [ ]:
class Generator:
    """Continues prompts with the KV cache; batches are shared between all GPUs (one model copy each)."""

    def __init__(self, s, model, tokenizer):
        self.s, self.tok = s, tokenizer
        devices = [torch.device(f"cuda:{i}") for i in range(N_GPU)] or [torch.device("cpu")]
        model.eval()
        self.copies = [(model if d == next(model.parameters()).device else copy.deepcopy(model).to(d).eval(), d)
                       for d in devices]

    def _sample(self, logits, gen, d):
        logits = logits.float(); logits[:, SPECIAL] = -float("inf")
        if d["temperature"] <= 0: return logits.argmax(-1, keepdim=True)
        logits = logits / d["temperature"]
        if d["top_k"]:
            kth = torch.topk(logits, min(d["top_k"], logits.size(-1))).values[:, -1:]
            logits = logits.masked_fill(logits < kth, -float("inf"))
        probs = F.softmax(logits, -1)
        if d["top_p"] < 1.0:
            p, order = probs.sort(-1, descending=True)
            p = p * ((p.cumsum(-1) - p) <= d["top_p"]); p = p / p.sum(-1, keepdim=True)
            return order.gather(-1, torch.multinomial(p, 1, generator=gen))
        return torch.multinomial(probs, 1, generator=gen)

    @torch.no_grad()
    def _batch(self, net, dev, encoded, seed, d):
        s = self.s; room = s.ctx - s.prompt_room
        P = min(min(map(len, encoded)), room)
        ids = torch.tensor([e[-P:] for e in encoded], device=dev)
        gen = torch.Generator(device=dev); gen.manual_seed(seed)
        with autocast():
            logits, past = net.prefill(ids)
        start = 0
        for t in range(s.max_new):
            nxt = self._sample(logits, gen, d); ids = torch.cat([ids, nxt], 1)
            pos = ids.size(1) - 1 - start
            with autocast():
                if pos >= s.ctx:                          # cache full: re-read the recent text and go on
                    start = ids.size(1) - room; logits, past = net.prefill(ids[:, start:])
                else:
                    logits, past = net.step(nxt, past, pos)
            if (t + 1) % 16 == 0 and all(len(self.tok.decode(r).split()) > s.words for r in ids[:, P:].tolist()):
                break
        return [" ".join(self.tok.decode(r).split()[:s.words]) for r in ids[:, P:].tolist()]

    def __call__(self, prompts, decoding=None, seed=0):
        """prompts: {id: {"text": ...}} -> {id: {"text": 128-word continuation}}"""
        s = self.s
        d = decoding or dict(temperature=s.temperature, top_k=s.top_k_sample, top_p=s.top_p)
        enc = {k: self.tok.encode(v["text"].rstrip()) for k, v in prompts.items()}
        order = sorted(enc, key=lambda k: len(enc[k]))
        jobs = queue.Queue()
        for b in range(0, len(order), s.gen_batch): jobs.put(b)
        out, errors, lock, t0 = {}, [], threading.Lock(), time.time()

        def work(net, dev):
            try:
                while True:
                    try: b = jobs.get_nowait()
                    except queue.Empty: return
                    keys = order[b:b + s.gen_batch]
                    texts = self._batch(net, dev, [enc[k] for k in keys], s.seed + seed + b, d)
                    with lock: out.update({k: {"text": t} for k, t in zip(keys, texts)})
            except Exception as e:
                errors.append(e)

        threads = [threading.Thread(target=work, args=c, daemon=True) for c in self.copies]
        for th in threads: th.start()
        while any(th.is_alive() for th in threads):
            time.sleep(30)
            if len(prompts) > 200: print(f"  {len(out)}/{len(prompts)} generated ({time.time()-t0:.0f}s)")
        if errors: raise errors[0]
        return {k: out[k] for k in prompts}

## 7 · Submission helpers

In [ ]:
def complete(generator, prompts, preds, decoding=None, tries=3):
    """Re-generate (new seed) the few continuations that came out shorter than 128 words."""
    for attempt in range(1, tries + 1):
        short = [k for k, v in preds.items() if len(v["text"].split()) < S.words]
        if not short: break
        redo = generator({k: prompts[k] for k in short}, decoding, seed=1000 * attempt)
        for k in short:
            if len(redo[k]["text"].split()) > len(preds[k]["text"].split()): preds[k] = redo[k]
    return preds


def write_submission(preds, test_ids, folder, name):
    assert set(preds) == set(test_ids), "IDs must match the test set exactly"
    assert all(isinstance(v["text"], str) and v["text"].strip() for v in preds.values())
    path = os.path.join(folder, "prediction.json")
    json.dump(preds, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    zpath = os.path.join(folder, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"] and set(json.loads(z.read("prediction.json"))) == set(test_ids)
    words = [len(v["text"].split()) for v in preds.values()]
    print(f"{len(preds)} continuations, words {min(words)}-{max(words)} | verified -> {zpath}")
    return zpath


_SCORER = {}
def metrics(preds, refs, bertscore=True):
    keys = list(refs); hyp = [preds[k]["text"] for k in keys]; ref = [refs[k]["text"] for k in keys]; res = {}
    from rouge_score import rouge_scorer
    rs = rouge_scorer.RougeScorer(["rougeL"])
    res["rougeL"] = float(np.mean([rs.score(r, h)["rougeL"].fmeasure for r, h in zip(ref, hyp)]))
    import sacrebleu
    res["bleu4"] = sacrebleu.corpus_bleu(hyp, [ref]).score / 100
    if bertscore:
        try:
            if "bs" not in _SCORER:
                from bert_score import BERTScorer
                _SCORER["bs"] = BERTScorer(lang="en", batch_size=16)
            res["bertscore"] = float(_SCORER["bs"].score(hyp, ref)[2].mean())
        except Exception as e:
            print("BERTScore skipped:", e)
    return {k: round(v, 4) for k, v in res.items()}


def make_bundle(run_dir, keep_last=True):
    path = os.path.join(S.out, os.path.basename(run_dir) + "_bundle.zip")
    with zipfile.ZipFile(path, "w") as z:
        for root, _, files in os.walk(run_dir):
            if os.path.abspath(root).startswith(os.path.abspath(DATA_DIR)): continue   # token files stay out
            for f in files:
                if f.endswith(".tmp") or (f == "last.pt" and not keep_last): continue
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, os.path.dirname(run_dir)),
                        compress_type=zipfile.ZIP_STORED if f.endswith(".pt") else zipfile.ZIP_DEFLATED)
        for f in os.listdir("src"):
            if f.endswith(".py"): z.write(os.path.join("src", f), os.path.join(os.path.basename(run_dir), "src", f))
    print(f"bundle -> {path} ({os.path.getsize(path)/1e6:.0f} MB)")
    return path

## 8 · Run

In [ ]:
# 8a · corpus -> tokenizer -> token files (skipped when this run folder already has them)
store = TokenStore(S, DATA_DIR)
if not store.ready():
    corpus = LegalCorpus(S)
    train_docs, held_docs = corpus.build()
    store.build(train_docs, held_docs, LegalCorpus.dev_set(held_docs, S.dev_prompts, S.seed))
    del train_docs, held_docs
tokenizer, train_ids, val_ids, meta, (dev_prompts, dev_refs) = store.load()
print(f"train {len(train_ids):,} tokens | val {len(val_ids):,} | chars/token {meta['chars_per_token']} | "
      f"dev prompts {len(dev_prompts)} | tokenizer {meta['tokenizer']}")
print("train tokens by source:", meta["train_tokens_by_source"])

In [ ]:
# 8b · model + training
cfg = MoEConfig(vocab_size=tokenizer.get_vocab_size(), d_model=S.d_model, n_layers=S.n_layers, n_heads=S.n_heads,
                n_kv_heads=S.n_kv_heads, n_experts=S.n_experts, top_k=S.top_k, expert_hidden=S.expert_hidden,
                max_seq_len=S.ctx, aux_coef=S.aux_coef)
model = MoELanguageModel(cfg)
total, active = model.param_counts()
print(f"parameters: {total/1e6:.1f}M total, {active/1e6:.1f}M active per token")
if not S.smoke:
    assert 50e6 <= total <= 300e6, "Phase-1 models must have 50M-300M parameters"
trainer = Trainer(S, model, train_ids, val_ids, meta, RUN_DIR, meta["tokenizer"])
last = os.path.join(RUN_DIR, "last.pt")
if S.resume or os.path.exists(last):
    trainer.resume(last if os.path.exists(last) else S.resume)
else:
    print(f"initial val loss {trainer.val_loss(sum(trainer.windows.values(), [])):.3f} "
          f"(ln V = {math.log(cfg.vocab_size):.3f})")
trainer.fit()
print(f"training finished at step {trainer.step}: best val {trainer.best:.3f} | session {hours():.2f} h")
with torch.no_grad():
    model.eval(); model(trainer._batch(val_ids, sum(trainer.windows.values(), [])[:S.micro_batch])[0])
print("expert load per layer (share of routed tokens, ideal", round(1 / cfg.n_experts, 3), "):")
for i, row in enumerate(model.expert_load().tolist()): print(f"  layer {i:>2}: " + " ".join(f"{v:.3f}" for v in row))

In [ ]:
# 8c · learning curve
import matplotlib.pyplot as plt
h = trainer.history
if h:
    plt.figure(figsize=(7, 3.5))
    plt.plot([r["step"] for r in h], [r["raw"] for r in h], "o-", label="raw weights")
    if h[0]["ema"] is not None: plt.plot([r["step"] for r in h], [r["ema"] for r in h], "s-", label="EMA weights")
    plt.xlabel("step"); plt.ylabel("validation loss"); plt.title(S.run); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "curve.png"), dpi=120); plt.show()

In [ ]:
# 8d · best weights -> 1000 test continuations -> submission.zip
best = torch.load(os.path.join(RUN_DIR, "best.pt"), map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model"]); model.eval()
print(f"best checkpoint: step {best['step']}, val {best['val']:.3f} ({best['weights']} weights)")
generator = Generator(S, model, tokenizer)
test = LegalCorpus(S).test_prompts()
t0 = time.time()
predictions = complete(generator, test, generator(test))
print(f"generated in {(time.time()-t0)/60:.1f} min")
zip_path = write_submission(predictions, test.keys(), RUN_DIR, S.run)

In [ ]:
# 8e · local evaluation on the held-out dev set, run record, download bundle
dev_preds = generator(dev_prompts)
dev_scores = metrics(dev_preds, dev_refs)
by_source = {src: metrics(dev_preds, {k: v for k, v in dev_refs.items() if v["source"] == src}, bertscore=False)
             for src in S.sources}
record = dict(settings=asdict(S), params_total=total, params_active=active, best_step=best["step"], best_val=best["val"],
              val_by_source=trainer.val_by_source(), dev=dev_scores, dev_by_source=by_source, data=meta,
              hours=round(hours(), 2), expert_load=model.expert_load().tolist())
json.dump(record, open(os.path.join(RUN_DIR, "record.json"), "w"), indent=1)
print("dev:", dev_scores); print("val loss by source:", record["val_by_source"])
k0 = next(iter(dev_refs)); print("\nREF :", dev_refs[k0]["text"][:250]); print("PRED:", dev_preds[k0]["text"][:250])
make_bundle(RUN_DIR, keep_last=False)       # last.pt (~4 GB) stays in the run folder for resuming
print(f"\nupload {zip_path} to Codabench")